# Bonus — Flappy Bird: l'agente impara a volare

Flappy Bird è un esempio perfetto di Reinforcement Learning applicato a un vero videogioco, vicino allo spirito del progetto Pokémon.

L'agente vede 12 numeri (posizione dell'uccellino, distanza dai tubi, ecc.) e ha 2 azioni: battere le ali o non fare nulla. Impara da solo a sopravvivere e passare i tubi.

> Con ~300.000 passi l'agente inizia a passare qualche tubo (circa 3 minuti su Colab con GPU). Più lo allenate, meglio gioca. Su CPU è più lento: se andate di fretta, riducete i passi.

In [1]:
!pip install flappy-bird-gymnasium stable-baselines3 --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import gymnasium as gym
import flappy_bird_gymnasium
from stable_baselines3 import DQN
import numpy as np

# use_lidar=False -> l'agente riceve 12 numeri (non i pixel): impara molto piu in fretta
env = gym.make("FlappyBird-v0", use_lidar=False)

print("Cosa vede l'agente:", env.observation_space.shape[0], "numeri")
print("Azioni possibili:", env.action_space.n, "(0 = non fare nulla, 1 = batti le ali)")

c:\Users\LuciaGasperini\.virtualenvs\ifab\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Cosa vede l'agente: 12 numeri
Azioni possibili: 2 (0 = non fare nulla, 1 = batti le ali)


In [3]:
# Funzione per contare quanti TUBI passa l'agente (metrica chiara)
def conta_tubi(modello, episodi=15):
    tubi = []
    for _ in range(episodi):
        obs, _ = env.reset()
        done = False; passati = 0; passi = 0
        while not done and passi < 5000:
            azione, _ = modello.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(int(azione))
            if r >= 1.0:          # ricompensa grande = tubo superato
                passati += 1
            done = term or trunc; passi += 1
        tubi.append(passati)
    return np.mean(tubi), np.max(tubi)

In [4]:
# Costruiamo l'agente DQN (Deep Q-Network): una rete neurale al posto della tabella Q
modello_fb = DQN(
    "MlpPolicy", env, verbose=0,
    learning_rate=5e-4,
    buffer_size=100000,
    learning_starts=5000,
    batch_size=128,
    gamma=0.99,
    exploration_fraction=0.3,    # esplora molto all'inizio, poi sfrutta
    exploration_final_eps=0.01,
    train_freq=4,
    target_update_interval=2000,
)

# Prima dell'addestramento: quanti tubi passa? (praticamente zero)
media_pre, _ = conta_tubi(modello_fb)
print(f"PRIMA dell'addestramento: {media_pre:.1f} tubi in media (cade subito)")

PRIMA dell'addestramento: 0.0 tubi in media (cade subito)


In [5]:
# Addestriamo. 300.000 passi: circa 3 minuti su Colab con GPU.
# Suggerimento: su Colab attivate la GPU da  Runtime > Cambia tipo di runtime > GPU
modello_fb.learn(total_timesteps=300000)

media_post, best = conta_tubi(modello_fb)
print(f"DOPO l'addestramento: {media_post:.1f} tubi in media, record {best:.0f} tubi")
print("L'agente ha imparato a volare da solo, solo a forza di provare e sbagliare!")

DOPO l'addestramento: 1.1 tubi in media, record 4 tubi
L'agente ha imparato a volare da solo, solo a forza di provare e sbagliare!


### Guardiamo l'agente giocare
Registriamo una partita dell'agente addestrato e la mostriamo come animazione.

In [6]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Ambiente video per registrare la partita
env_video = gym.make("FlappyBird-v0", use_lidar=False, render_mode="rgb_array")

# Registriamo il miglior tentativo tra 8 (vogliamo vedere almeno 2 tubi)
best_frames, best_tubi = [], 0
for attempt in range(8):
    obs, _ = env_video.reset()
    ep_frames, tubi = [], 0
    done = False; passi = 0
    while not done and passi < 3000:
        azione, _ = modello_fb.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = env_video.step(int(azione))
        ep_frames.append(env_video.render())
        if r >= 1.0:
            tubi += 1
        done = term or trunc; passi += 1
    if tubi > best_tubi:
        best_tubi, best_frames = tubi, ep_frames
    if best_tubi >= 2:
        break

env_video.close()
frames = best_frames
print(f"Miglior tentativo: {best_tubi} tubi, {len(frames)} fotogrammi")

# Animazione (un fotogramma ogni 2 per alleggerire)
fig = plt.figure(figsize=(4, 6))
plt.axis('off')
immagine = plt.imshow(frames[0])
def aggiorna(i):
    immagine.set_array(frames[min(i*2, len(frames)-1)])
    return [immagine]
anim = animation.FuncAnimation(fig, aggiorna, frames=len(frames)//2, interval=40, blit=True)
plt.close()
HTML(anim.to_jshtml())

Miglior tentativo: 2 tubi, 131 fotogrammi


**Lo stesso identico principio del progetto Pokémon**, ma su un gioco semplice: un agente, un ambiente (il gioco), ricompense (sopravvivere e passare i tubi), e una rete neurale che impara la strategia.

La differenza con Pokémon è solo la **complessità dell'ambiente**: Flappy Bird ha 12 numeri e 2 azioni, Pokémon ha lo schermo intero e decine di azioni possibili — per questo lì servono milioni di passi e una GPU potente.

### Provate voi
- Allenate più a lungo (`total_timesteps=600000`): quanti tubi passa ora?
- Provate a cambiare `exploration_fraction`: cosa succede se esplora troppo poco?
- Confrontate con un compagno: chi ha allenato l'agente migliore?